### Importing the libraries

In [20]:
import numpy as np
import pandas as pd
import tensorflow as tf

In [21]:
tf.__version__

'2.11.1'

## Part 1 - Data Preprocessing

### Importing the dataset

In [22]:
dataset = pd.read_excel("Folds5x2_pp.xlsx")
X = dataset.iloc[:,:-1].values
y = dataset.iloc[:,-1].values
print(X)
print(y)

[[  14.96   41.76 1024.07   73.17]
 [  25.18   62.96 1020.04   59.08]
 [   5.11   39.4  1012.16   92.14]
 ...
 [  31.32   74.33 1012.92   36.48]
 [  24.48   69.45 1013.86   62.39]
 [  21.6    62.52 1017.23   67.87]]
[463.26 444.37 488.56 ... 429.57 435.74 453.28]


### Splitting the dataset into the Training set and Test set

In [23]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)

## Part 2 - Building the ANN

### Initializing the ANN

In [24]:
ann = tf.keras.models.Sequential() #sınıfın misrascısını tanımladık

### Adding the input layer and the first hidden layer

In [25]:
ann.add(tf.keras.layers.Dense(units=6, activation='relu'))
#aslında burada girişleri tanımlamıyoruz zaten veriyi aktarırken otomatik olarak alıyor

### Adding the second hidden layer

In [26]:
ann.add(tf.keras.layers.Dense(units=6, activation='relu'))

### Adding the output layer

In [27]:
ann.add(tf.keras.layers.Dense(units=1))
#regression yaptığımız için none olarak bıraktık activasyon fonksiyonunu

## Part 3 - Training the ANN

### Compiling the ANN

In [28]:
ann.compile(optimizer = 'adam', loss = 'mean_squared_error')
#

### Training the ANN model on the Training set

In [29]:
ann.fit(X_train, y_train, batch_size = 32, epochs = 100)

Epoch 1/100
240/240 [==============================] - 1s 1ms/step - loss: 450045.4688
Epoch 2/100
240/240 [==============================] - 0s 1ms/step - loss: 774.6547
Epoch 3/100
240/240 [==============================] - 0s 1ms/step - loss: 711.1840
Epoch 4/100
240/240 [==============================] - 0s 1ms/step - loss: 688.8983
Epoch 5/100
240/240 [==============================] - 0s 1ms/step - loss: 661.8071
Epoch 6/100
240/240 [==============================] - 0s 1ms/step - loss: 632.4437
Epoch 7/100
240/240 [==============================] - 0s 1ms/step - loss: 598.8907
Epoch 8/100
240/240 [==============================] - 0s 1ms/step - loss: 562.9432
Epoch 9/100
240/240 [==============================] - 0s 1ms/step - loss: 523.8684
Epoch 10/100
240/240 [==============================] - 0s 1ms/step - loss: 482.7072
Epoch 11/100
240/240 [==============================] - 0s 1ms/step - loss: 440.8422
Epoch 12/100
240/240 [==============================] - 0s 1ms/step - l

Tamam, bu eğitim çıktısını sana adım adım açıklayayım.
Bu iki konuyu anlamak önemli: **240/240** ifadesi nereden geliyor ve **batch\_size** aslında ne işe yarıyor.

---

## 1️⃣ **240/240 ne demek?**

Sen modeli şu şekilde eğitiyorsun:

```python
ann.fit(X_train, y_train, batch_size = 32, epochs = 100)
```

Eğitim setindeki örnek sayısı:

```
len(X_train) ≈ 7680
```

(Çünkü veri setinde 10.000 satır var, %80 train → 8000 civarı, encoding ve scaling sonrası biraz değişmiş olabilir.)

### **Batch mantığı**

* **Batch size = 32** demek:
  1 güncelleme (gradient descent adımı) için **32 örnek** kullan demek.
* Kaç batch oluşur?

```
Toplam batch sayısı = Toplam örnek sayısı / Batch size
                     ≈ 7680 / 32
                     = 240 batch
```

Yani **her epoch**’ta **240 kez ağırlık güncellemesi** yapılır.
Bunu eğitim sırasında **"240/240"** şeklinde gösteriyor.

📌 Yani:

* **240** = Bu epoch'ta kaç batch var
* **/240** = Toplam batch sayısı
  → İkisi aynı olunca, bu epoch'un son batch'ini işliyor demek.

---

## 2️⃣ **Batch size Gradient Descent ile nasıl ilgili?**

Gradient Descent’in türleri vardı:

* **Batch GD** → Tüm veri ile bir güncelleme
* **Stochastic GD (SGD)** → 1 örnek ile bir güncelleme
* **Mini-batch GD** → Küçük grup (örneğin 32 örnek) ile bir güncelleme ✅ *(Seninki bu)*

Sen burada **Mini-Batch Gradient Descent** yapıyorsun.
Batch size = 32 → Her adımda 32 örnek üzerinden loss hesaplanır ve ağırlıklar güncellenir.

---

## 3️⃣ **Neden 32 seçiyoruz?**

* **32** makine öğrenmesinde ve derin öğrenmede **çok yaygın bir default değer**.
* GPU’larda **hafıza ve işlem hızını** optimize eder (2’nin katı: 16, 32, 64 genelde iyi çalışır).
* 32 çoğu veri setinde **hız + kararlılık** açısından iyi dengedir.

Küçük batch → Daha gürültülü öğrenme, ama daha hızlı genelleme.
Büyük batch → Daha kararlı, ama daha yavaş yakınsama.

---

## 📌 Özet

* **240/240** → 240 batch var, o epoch’un 240. batch’indeyiz.
* **Batch size** = 32 → Mini-Batch Gradient Descent yapıyorsun.
* 32 → Performans + kararlılık dengesi için standart bir değer.

---

Eğer istersen ben sana **batch size değiştirince eğitim hızının ve loss grafiğinin nasıl değiştiğini** görsel olarak gösterebilirim.
O zaman 16, 32, 64 farkını gözünle görürsün.
İster misin bunu yapayım?


### Predicting the results of the Test set

In [31]:
y_pred = ann.predict(X_test)
np.set_printoptions(precision=2)
print(np.concatenate((y_pred.reshape(len(y_pred),1), y_test.reshape(len(y_test),1)),1))
#sol taraf tahminler sağ taraf gerçek değerler

60/60 [==============================] - 0s 1ms/step
[[430.18 431.23]
 [461.17 460.01]
 [464.66 461.14]
 ...
 [471.91 473.26]
 [438.67 438.  ]
 [457.9  463.28]]
